# AgriGuard AI — train the pest detector on IP102 (Colab GPU)

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Allow Google Drive access when asked (checkpoints are saved there so a disconnect doesn't lose training).
4. When the last cell finishes, `agriguard_model.zip` downloads automatically. Leave it in your Downloads folder.

Dataset: IP102 detection subset (Wu et al., CVPR 2019), official Google Drive mirror linked from https://github.com/xpwu95/IP102. Model: Ultralytics YOLO11s fine-tuned from COCO weights. Expected time on a T4: about 2–3 hours.

In [ ]:
!nvidia-smi -L
import torch; assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/agriguard_runs

In [ ]:
!pip -q install ultralytics==8.3.* gdown onnx onnxslim onnxruntime
!mkdir -p /content/ag

In [ ]:
%%writefile /content/ag/prepare_ip102.py
"""Download, verify, clean and convert the IP102 detection subset to YOLO format.

Source: IP102 (Wu et al., CVPR 2019), official Google Drive folder linked from
https://github.com/xpwu95/IP102 . No explicit licence file is published; the
authors ask for citation. Use for research/education and cite the paper.

Steps (all re-runnable):
 1. Download only the Detection files from the official Drive folder (gdown).
 2. Extract the tar archives.
 3. Parse Pascal-VOC XML annotations; map numeric labels to classes.txt names.
 4. Validate: unreadable/corrupt images, empty or out-of-bounds boxes, unknown labels.
 5. Remove exact duplicate images (SHA-1) — duplicates are kept in ONE split only.
 6. Split using the official trainval/test lists when present (val carved from
    trainval with a fixed seed), otherwise a stratified 80/10/10 split.
 7. Write YOLO labels, data.yaml and dataset_stats.json.

Usage:  python prepare_ip102.py --work /content/ip102 --out /content/ip102_yolo
"""
import argparse
import hashlib
import json
import random
import shutil
import subprocess
import sys
import tarfile
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

DRIVE_FOLDER = "https://drive.google.com/drive/folders/1svFSy2Da3cVMvekBwe13mzyx38XZ9xWo"
CLASSES_URL = "https://raw.githubusercontent.com/xpwu95/IP102/master/classes.txt"
SEED = 42


def sh(cmd):
    print("+", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


def load_classes(work: Path) -> list[str]:
    work.mkdir(parents=True, exist_ok=True)
    p = work / "classes.txt"
    if not p.exists():
        sh(f"wget -q -O {p} {CLASSES_URL}")
    names = []
    for line in p.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line:
            continue
        idx, name = line.split(maxsplit=1)
        names.append(" ".join(name.split()))
    assert len(names) == 102, f"expected 102 classes, got {len(names)}"
    return names


def download(work: Path):
    import gdown  # noqa
    work.mkdir(parents=True, exist_ok=True)
    files = gdown.download_folder(DRIVE_FOLDER, output=str(work / "drive"), skip_download=True, quiet=True)
    det = [f for f in files if "Detection" in f.path or "detection" in f.path.lower()]
    print("Drive files:", [f.path for f in files])
    print("Detection files selected:", [f.path for f in det])
    if not det:
        sys.exit("No Detection files found in the Drive folder — inspect the listing above.")
    for f in det:
        dest = Path(f.local_path)
        if dest.exists() and dest.stat().st_size > 0:
            print("exists:", dest)
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        gdown.download(id=f.id, output=str(dest), quiet=False)
        if not dest.exists():
            sys.exit(f"Download failed for {f.path} (Drive quota?). Retry later.")
    for tar in (work / "drive").rglob("*.tar*"):
        marker = tar.with_suffix(".extracted")
        if marker.exists():
            continue
        print("extracting", tar)
        with tarfile.open(tar) as t:
            t.extractall(tar.parent / "extracted", filter="data")
        marker.touch()


def parse_voc(xml_path: Path):
    root = ET.parse(xml_path).getroot()
    fname = root.findtext("filename")
    size = root.find("size")
    w = int(float(size.findtext("width"))) if size is not None else 0
    h = int(float(size.findtext("height"))) if size is not None else 0
    objs = []
    for o in root.iter("object"):
        bb = o.find("bndbox")
        objs.append((o.findtext("name").strip(),
                     float(bb.findtext("xmin")), float(bb.findtext("ymin")),
                     float(bb.findtext("xmax")), float(bb.findtext("ymax"))))
    return fname, w, h, objs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--work", default="/content/ip102")
    ap.add_argument("--out", default="/content/ip102_yolo")
    ap.add_argument("--skip-download", action="store_true")
    a = ap.parse_args()
    work, out = Path(a.work), Path(a.out)
    names = load_classes(work)
    if not a.skip_download:
        download(work)

    from PIL import Image

    images = {p.name: p for p in work.rglob("*.jpg")}
    xmls = list(work.rglob("*.xml"))
    print(f"found {len(images)} jpg, {len(xmls)} xml")

    # Detect whether numeric labels are 0- or 1-based.
    raw_labels = Counter()
    parsed = []
    for x in xmls:
        try:
            parsed.append((x, *parse_voc(x)))
        except Exception as e:  # malformed XML
            print("bad xml", x, e)
    for _, _, _, _, objs in parsed:
        raw_labels.update(o[0] for o in objs)
    numeric = all(l.isdigit() for l in raw_labels)
    offset = 0
    if numeric:
        vals = sorted(int(l) for l in raw_labels)
        offset = 1 if vals[0] >= 1 and vals[-1] == 102 else 0
        print(f"numeric labels {vals[0]}..{vals[-1]} -> offset {offset}")
    name_to_idx = {n.lower(): i for i, n in enumerate(names)}

    stats = Counter()
    records = {}  # stem -> (img_path, w, h, [(cls, cx, cy, bw, bh)])
    hashes = {}
    for x, fname, w, h, objs in parsed:
        stem = Path(fname or x.stem).stem
        img = images.get(f"{stem}.jpg") or images.get(fname or "")
        if img is None:
            stats["missing_image"] += 1
            continue
        try:
            with Image.open(img) as im:
                im.verify()
            with Image.open(img) as im:
                w, h = im.size
        except Exception:
            stats["corrupt_image"] += 1
            continue
        digest = hashlib.sha1(img.read_bytes()).hexdigest()
        if digest in hashes:
            stats["duplicate_image"] += 1
            continue
        hashes[digest] = stem
        boxes = []
        for lab, x1, y1, x2, y2 in objs:
            if numeric:
                ci = int(lab) - offset
            else:
                ci = name_to_idx.get(lab.lower(), -1)
            if not 0 <= ci < len(names):
                stats["invalid_label"] += 1
                continue
            x1, x2 = max(0, min(x1, x2)), min(w, max(x1, x2))
            y1, y2 = max(0, min(y1, y2)), min(h, max(y1, y2))
            if x2 - x1 < 2 or y2 - y1 < 2:
                stats["degenerate_box"] += 1
                continue
            boxes.append((ci, (x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h))
        if not boxes:
            stats["no_valid_boxes"] += 1
            continue
        records[stem] = (img, w, h, boxes)
    print("cleaning stats:", dict(stats), "kept:", len(records))

    # Official split lists if present
    def read_list(pattern):
        for p in work.rglob(pattern):
            return {l.strip() for l in p.read_text().split() if l.strip()}
        return None

    trainval, test = read_list("trainval.txt"), read_list("test.txt")
    rnd = random.Random(SEED)
    split = {}
    if trainval and test:
        tv = sorted(s for s in records if s in trainval)
        rnd.shuffle(tv)
        n_val = max(1, int(0.1 * len(tv)))
        for s in tv[:n_val]:
            split[s] = "val"
        for s in tv[n_val:]:
            split[s] = "train"
        for s in records:
            if s in test:
                split[s] = "test"
        split_method = "official IP102 trainval/test lists; val = 10% of trainval (seed 42)"
    else:
        by_cls = defaultdict(list)
        for s, (_, _, _, boxes) in records.items():
            by_cls[boxes[0][0]].append(s)
        for cls, stems in by_cls.items():
            stems.sort(); rnd.shuffle(stems)
            n = len(stems); nt = int(0.8 * n); nv = int(0.1 * n)
            for i, s in enumerate(stems):
                split[s] = "train" if i < nt else ("val" if i < nt + nv else "test")
        split_method = "stratified 80/10/10 by first-box class (seed 42)"

    if out.exists():
        shutil.rmtree(out)
    per_split_imgs = Counter()
    per_class = {k: Counter() for k in ("train", "val", "test")}
    for s, (img, w, h, boxes) in records.items():
        sp = split.get(s)
        if sp is None:
            continue
        (out / "images" / sp).mkdir(parents=True, exist_ok=True)
        (out / "labels" / sp).mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, out / "images" / sp / f"{s}.jpg")
        (out / "labels" / sp / f"{s}.txt").write_text(
            "\n".join(f"{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}" for c, cx, cy, bw, bh in boxes))
        per_split_imgs[sp] += 1
        per_class[sp].update(b[0] for b in boxes)

    yaml_names = "\n".join(f"  {i}: \"{n}\"" for i, n in enumerate(names))
    (out / "data.yaml").write_text(
        f"path: {out}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(names)}\nnames:\n{yaml_names}\n")
    result = {
        "dataset": "IP102 detection subset (v1.1)",
        "source": DRIVE_FOLDER,
        "homepage": "https://github.com/xpwu95/IP102",
        "citation": "Wu et al., IP102: A Large-Scale Benchmark Dataset for Insect Pest Recognition, CVPR 2019",
        "licence": "No explicit licence published by the authors; research use with citation.",
        "classes": names,
        "split_method": split_method,
        "cleaning": dict(stats),
        "images_found": len(images), "annotations_found": len(xmls),
        "images_per_split": dict(per_split_imgs),
        "boxes_per_class": {sp: {names[c]: n for c, n in sorted(cnt.items())} for sp, cnt in per_class.items()},
    }
    (out / "dataset_stats.json").write_text(json.dumps(result, indent=2))
    print(json.dumps({k: v for k, v in result.items() if k != "boxes_per_class"}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/ag/train.py
"""Fine-tune a YOLO detector on the prepared IP102 YOLO dataset.

Usage:  python train.py --data /content/ip102_yolo/data.yaml --model yolo11s.pt --epochs 40
Outputs: runs/agriguard/<name>/weights/best.pt plus training logs (results.csv).
Resume after a Colab disconnect:  python train.py --resume runs/agriguard/<name>/weights/last.pt
"""
import argparse

from ultralytics import YOLO


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="/content/ip102_yolo/data.yaml")
    ap.add_argument("--model", default="yolo11s.pt")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--project", default="runs/agriguard")
    ap.add_argument("--name", default="ip102_yolo11s")
    ap.add_argument("--resume", default="")
    a = ap.parse_args()

    if a.resume:
        YOLO(a.resume).train(resume=True)
        return
    model = YOLO(a.model)  # COCO-pretrained weights -> transfer learning
    model.train(
        data=a.data, epochs=a.epochs, imgsz=a.imgsz, batch=a.batch,
        project=a.project, name=a.name, exist_ok=True,
        patience=15, save_period=5, seed=42, deterministic=False,
        cos_lr=True, close_mosaic=5, workers=4, plots=True,
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/ag/evaluate.py
"""Evaluate trained weights on the held-out TEST split and write model_meta.json.

Reports precision, recall, F1, mAP@0.5 and mAP@0.5:0.95 (overall and per class)
and saves the confusion matrix produced by Ultralytics.

Usage: python evaluate.py --weights best.pt --data data.yaml --stats dataset_stats.json --out eval_out
"""
import argparse
import json
import shutil
import time
from pathlib import Path

from ultralytics import YOLO


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--stats", default="")
    ap.add_argument("--out", default="eval_out")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--version", default="1.0.0")
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

    model = YOLO(a.weights)
    t = time.time()
    r = model.val(data=a.data, split="test", imgsz=a.imgsz, batch=32, plots=True,
                  project=str(out), name="test", exist_ok=True)
    names = model.names
    p, rc = float(r.box.mp), float(r.box.mr)
    f1 = 2 * p * rc / (p + rc) if p + rc else 0.0
    per_class = {}
    for i, c in enumerate(r.box.ap_class_index):
        c = int(c)
        pc, rcc = float(r.box.p[i]), float(r.box.r[i])
        per_class[names[c]] = {
            "precision": round(pc, 4), "recall": round(rcc, 4),
            "f1": round(2 * pc * rcc / (pc + rcc), 4) if pc + rcc else 0.0,
            "map50": round(float(r.box.ap50[i]), 4), "map50_95": round(float(r.box.ap[i]), 4),
        }
    metrics = {
        "split": "test",
        "precision": round(p, 4), "recall": round(rc, 4), "f1": round(f1, 4),
        "map50": round(float(r.box.map50), 4), "map50_95": round(float(r.box.map), 4),
        "speed_ms_per_image": {k: round(v, 2) for k, v in r.speed.items()},
        "eval_seconds": round(time.time() - t, 1),
        "per_class": per_class,
    }
    stats = json.loads(Path(a.stats).read_text()) if a.stats else {}
    meta = {
        "name": "AgriGuard Pest Detector",
        "version": a.version,
        "architecture": f"Ultralytics {Path(a.weights).name} ({model.model.yaml.get('yaml_file', 'YOLO')})",
        "dataset": stats.get("dataset", "IP102 detection subset"),
        "dataset_source": stats.get("homepage", ""),
        "classes": [names[i] for i in range(len(names))],
        "metrics": metrics,
        "dataset_stats": {k: v for k, v in stats.items() if k != "classes"},
    }
    (out / "model_meta.json").write_text(json.dumps(meta, indent=2))
    for f in (out / "test").glob("*.png"):
        shutil.copy2(f, out / f.name)
    print(json.dumps({k: v for k, v in metrics.items() if k != "per_class"}, indent=2))


if __name__ == "__main__":
    main()


## 1. Download, clean, convert and split IP102 (~15–25 min)

In [ ]:
!cd /content/ag && python prepare_ip102.py --work /content/ip102 --out /content/ip102_yolo

## 2. Train (resumes automatically from Drive if a previous session was cut off)

In [ ]:
import os
last = '/content/drive/MyDrive/agriguard_runs/ip102_yolo11s/weights/last.pt'
if os.path.exists(last):
    !cd /content/ag && python train.py --resume {last}
else:
    !cd /content/ag && python train.py --data /content/ip102_yolo/data.yaml --model yolo11s.pt --epochs 35 --batch 32 --project /content/drive/MyDrive/agriguard_runs --name ip102_yolo11s

## 3. Evaluate on the held-out TEST split

In [ ]:
!cd /content/ag && python evaluate.py --weights /content/drive/MyDrive/agriguard_runs/ip102_yolo11s/weights/best.pt --data /content/ip102_yolo/data.yaml --stats /content/ip102_yolo/dataset_stats.json --out /content/eval_out

## 4. Package and download

In [ ]:
import shutil, os
run = '/content/drive/MyDrive/agriguard_runs/ip102_yolo11s'
out = '/content/agriguard_model'
os.makedirs(out, exist_ok=True)
shutil.copy(f'{run}/weights/best.pt', out)
for f in ['results.csv', 'results.png', 'args.yaml']:
    if os.path.exists(f'{run}/{f}'): shutil.copy(f'{run}/{f}', out)
for f in os.listdir('/content/eval_out'):
    p = f'/content/eval_out/{f}'
    if os.path.isfile(p): shutil.copy(p, out)
shutil.copy('/content/ip102_yolo/dataset_stats.json', out)
from ultralytics import YOLO
onnx_path = YOLO(f'{out}/best.pt').export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.copy(onnx_path, f'{out}/best.onnx')
shutil.make_archive('/content/agriguard_model', 'zip', out)
shutil.copy('/content/agriguard_model.zip', '/content/drive/MyDrive/agriguard_runs/agriguard_model.zip')
print(os.listdir(out))
from google.colab import files
files.download('/content/agriguard_model.zip')